In [17]:
import rasterio
import numpy as np
from pathlib import Path
from dem_stitcher.rio_window import read_raster_from_window

In [ ]:
# logging Sweeden
xmin, ymin, xmax, ymax = (15.657, 61.4507,16.1521, 61.7201)
compare_dir = Path('gen_status_examples/logging__33VWJ/')
out_dir = Path('sweeden_logging_example')
out_dir.mkdir(exist_ok=True, parents=True)

# China Road Expansion
xmin, ymin, xmax, ymax = (119.8659, 35.8, 120.0207, 35.9508)
compare_dir = Path('gen_status_examples/road_expansion__50SQE/')
out_dir = Path('china_road_expansion_example')
out_dir.mkdir(exist_ok=True, parents=True)

## Indonesia Mining
xmin, ymin, xmax, ymax = (127.8628, 0.4446, 128.0627, 0.6385)
compare_dir = Path('gen_status_examples/mining__52NCF/')
out_dir = Path('indonesia_mining_example')
out_dir.mkdir(exist_ok=True, parents=True)

## Laos Shifting Cultivation
xmin, ymin, xmax, ymax = (105.9442, 16.5663, 106.4918, 16.9477)
compare_dir = Path('gen_status_examples/shifting_cultivation__48QXD/')
out_dir = Path('laos_shifting_cultivation_example')
out_dir.mkdir(exist_ok=True, parents=True)

# Shipping Area - Baltimore
xmin, ymin, xmax, ymax = -76.6517, 39.1746,  -76.4236, 39.3359
compare_dir = Path('gen_status_examples/new_construction__18SUJ/')
out_dir = Path('baltimore_shipping_area_example')
out_dir.mkdir(exist_ok=True, parents=True)


In [19]:
subset_dir = Path(f'subset/{compare_dir.stem}')
subset_dir.mkdir(exist_ok=True, parents=True)

rgb_dir = Path(f'rgb/{compare_dir.stem}')
rgb_dir.mkdir(exist_ok=True, parents=True)

In [20]:
all_tifs = list(Path(compare_dir).glob('*.tif'))
all_tifs 

[PosixPath('gen_status_examples/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_GEN-DIST-STATUS.tif'),
 PosixPath('gen_status_examples/new_construction__18SUJ/OPERA_L3_DIST-ALERT-S1_T18SUJ_20250727T230715Z_20260205T184019Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('gen_status_examples/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_VEG-DIST-STATUS.tif')]

In [21]:
bounds = (xmin, ymin, xmax, ymax)
def subset_one(tif):
    X, p = read_raster_from_window(tif, bounds) 
    out_tif = subset_dir / f'{tif.stem}_subset.tif'
    with rasterio.open(tif) as src:
        colormap = src.colormap(1)

    with rasterio.open(out_tif, 'w', **p) as dst:
        dst.write(X)
        dst.write_colormap(1, colormap)
    return out_tif

subset_tifs = [subset_one(tif) for tif in all_tifs]


In [22]:
dist_s1_tif = [tif for tif in subset_tifs if 'DIST-ALERT-S1' in tif.name][0]
dist_s1_tif

PosixPath('subset/new_construction__18SUJ/OPERA_L3_DIST-ALERT-S1_T18SUJ_20250727T230715Z_20260205T184019Z_S1A_30_v0.1_GEN-DIST-STATUS_subset.tif')

In [23]:
dist_hls_gen_tif = [tif for tif in subset_tifs if 'GEN-DIST-STATUS' in tif.name][0]
dist_hls_gen_tif

PosixPath('subset/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_GEN-DIST-STATUS_subset.tif')

In [24]:
dist_hls_veg_tif = [tif for tif in subset_tifs if 'VEG-DIST-STATUS' in tif.name][0]
dist_hls_veg_tif

PosixPath('subset/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_VEG-DIST-STATUS_subset.tif')

In [25]:
def indexed_to_rgb_robust(input_path, output_path):
    with rasterio.open(input_path) as src:
        colormap = src.colormap(1)
        if colormap is None:
            return

        indexed_data = src.read(1)

        max_index = max(colormap.keys())
        colormap_array = np.zeros((max_index + 1, 4), dtype=np.uint8)

        for index, rgba in colormap.items():
            colormap_array[index] = rgba

        rgb_rgba_data = colormap_array[indexed_data]

        rgb_data = rgb_rgba_data[:, :, :3].transpose(2, 0, 1)

        profile = src.profile
        profile.update(
            dtype=rasterio.uint8, 
            count=3,           
            nodata=None        
        )

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(rgb_data, indexes=[1, 2, 3])
            print(f"Successfully converted and saved RGB file to {output_path}")

In [26]:
from dist_s1.dist_plot import get_dist_s1_mpl_cmap



In [27]:
for tif in subset_tifs:
    rgb_out_tif = rgb_dir / f'{tif.stem}.tif'
    indexed_to_rgb_robust(tif, rgb_out_tif)

Successfully converted and saved RGB file to rgb/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_GEN-DIST-STATUS_subset.tif
Successfully converted and saved RGB file to rgb/new_construction__18SUJ/OPERA_L3_DIST-ALERT-S1_T18SUJ_20250727T230715Z_20260205T184019Z_S1A_30_v0.1_GEN-DIST-STATUS_subset.tif
Successfully converted and saved RGB file to rgb/new_construction__18SUJ/OPERA_L3_DIST-ALERT-HLS_T18SUJ_20250731T155819Z_20250813T013705Z_S2B_30_v1_VEG-DIST-STATUS_subset.tif


In [28]:
for tif in rgb_dir.glob('*.tif'):
    out_pmtiles = out_dir / f'{tif.stem}.pmtiles'
    !rio pmtiles {tif} {out_pmtiles} --format PNG --resampling nearest --zoom-levels 5..15


100%|███████████████████████████████████████████| 97/97 [00:01<00:00, 67.28it/s]
